IMPLEMENTAZIONE BACKPROPAGATION PYTORCH (AUTOGRAD)

PYTHORC
Immagina di dover tracciare un percorso in una foresta inesplorata. 
Invece di dover tracciare il percorso a mano, pythorc ci fornisce un GPS automatico
Pythorch si fornisce un GPS automatico che registra ogni nostro passo e che ci riporta al punto di partenza quando lo chiediamo.
Ma per attivare questo GPS dobbiamo prima dire a Pytorch quale sentieri deve monitorare.

- Il GPPS di Pythorch va attivato con requires_grad
- Autograd
- Come interrogare i parametri per vedere cosa hanno imparato.

COME PYTORCH COSTRUISCE LA SUA MEMORIA

In [6]:
import torch

In [ ]:
# 1. PREPARAZIONE DEI DATI
# Creiamo input (x) e target (y) come tensori costanti
x = torch.tensor([[1.5], [2.0], [3.0]], dtype=torch.float32) #3 dati in ingresso
y_target = torch.tensor([[0.5], [0.8], [1.2]], dtype=torch.float32) #3 risultati attesi

# 2. DEFINIZIONE DEI PARAMETRI (requires_grad=True)
# Inizializziamo il peso W e il bias b casualmente. 
# Fondamentale attivare il tracciamento dei gradienti.
#VANNO INZIALIZZAZI IN MODO CASUALE (casuale solo nella fase di inizializzazione, poi si aggiornano in modo deterministico)
W = torch.randn(1, 1, requires_grad=True) #requires_grad=True attiva il tracciamento dei gradienti per questo tensore, necessario per l ottimizzazione 
b = torch.randn(1, requires_grad=True)

print(f"Inizializzazione - W: {W.item():.4f}, b: {b.item():.4f}\n")

# 3. FORWARD PASS
# Costruiamo il grafo computazionale dinamico
y_pred = x @ W + b  # @=moltiplicazione matriciale
print(f"Predizioni (y_pred):\n{y_pred.detach().numpy()}")

# 4. CALCOLO DELLA LOSS
# Usiamo l'errore quadratico medio (MSE)
loss = torch.mean((y_pred - y_target)**2)
print(f"Loss calcolata: {loss.item():.6f}")

# 5. BACKWARD PASS (Motore Autograd)
# Calcola il gradiente di 'loss' rispetto a tutti i parametri con requires_grad=True
loss.backward() #attivo l'autograd, il pytorch attraverso il grafo a ritroso per calcolare i gradienti di W e b rispetto alla loss, questi gradienti vengono memorizzati negli attributi .grad di W e b

# 6. ISPEZIONE DEI GRADIENTI
# I gradienti vengono accumulati negli attributi .grad
print("\n--- Ispezione dei Gradienti ---")
print(f"Gradiente rispetto a W (dL/dW): {W.grad.item():.6f}") #.grad restituisce il gradiente calcolato per W, item() converte il tensore in un numero scalare
print(f"Gradiente rispetto a b (dL/db): {b.grad.item():.6f}")

# 7. AGGIORNAMENTO MANUALE (Teoria del Gradient Descent)
# Usiamo torch.no_grad() per evitare che l'operazione di aggiornamento 
# venga tracciata nel grafo (causerebbe errori o loop infiniti)
lr = 0.01
with torch.no_grad(): #.no_grad disabilita il tracciamento dei gradienti per le operazioni al suo interno, necessario per aggiornare i parametri senza interferire con il grafo computazionale
    W -= lr * W.grad
    b -= lr * b.grad

print(f"\nParametri dopo un passo di ottimizzazione:")
print(f"W: {W.item():.4f}, b: {b.item():.4f}")

Inizializzazione - W: 0.5171, b: -1.7329

Predizioni (y_pred):
[[-0.957284  ]
 [-0.6987473 ]
 [-0.18167377]]
Loss calcolata: 2.092981

--- Ispezione dei Gradienti ---
Gradiente rispetto a W (dL/dW): -6.218962
Gradiente rispetto a b (dL/db): -2.891804

Parametri dopo un passo di ottimizzazione:
W: 0.5793, b: -1.7040


TRACCIAMENTO AUTOGRAD
L'impostazione di requires_grad=True istruisce Pytorch a monitorare ogni operazione matematica eseguita sul tensore, costruendo in memoria il grafo computazionale necessario per la differenziazione auotmatica. Costruisce il libro mastro dei tensori. Ogni volta che sommate o moltiplicate i tensori con requires_grad attivo, pytorch scrive una riga nel libro mastro.
PROPAGAZIONE DELL'ERRORE
Chiamando .backward() su un tensore scalare (solitamente Loss), Pytorch attraversa il grafo a ritroso applicando la regola della catena per calcolare i gradienti rispetto a tutti i parametri tracciati.
Pytorch chiude illibro mastro e lo legge dall'ultima pagina alla prima, distribuendo le colpe dell'errore ad ogni peso. Ispezionare .grad è il primo strumento di debugging
ACCESSO AI GRADIENTI
I gradienti calcolati vengono memorizzati nell'attributo .grad di ogni tensore. E' fondamentale ispezionarli per diagnosticare problemi di vanisching o exploding gradienti prima dello step di ottimizzazione